# Magana: N-ATLaS on vLLM (development and demos only)

Serves **NCAIR1/N-ATLaS** with vLLM's OpenAI-compatible server, with `--enable-lora` so one GPU pair serves the base model **and** LoRA adapters: the Ibà adapter (`iba-atlas`) now, a Magana adapter later. The Magana backend reaches it through a temporary Cloudflare tunnel protected by an API key.

**Kaggle is for development and demos only** (docs/MODELS.md): sessions stop after at most about 12 hours (sooner when idle), GPU time is limited to about 30 hours a week, and the tunnel URL changes every session. Real users need a GPU server that stays up.

**Settings (right panel):** Accelerator **GPU T4 x2**, Internet **On**. **Secrets** (Add-ons → Secrets), ticked for this notebook:
- `HF_TOKEN`: a Hugging Face token from the account that accepted the N-ATLaS Terms of Use (and can read the private Ibà adapter).
- `EXPLAIN_API_KEY`: the key the tunnel requires; the same value as `EXPLAIN_API_KEY` in the Magana server's `.env`.

N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies. Use is limited to 1,000 active end-users per rolling 30 days without a commercial licence.

## 1. Setup (about 5 minutes)

In [ ]:
import os, re, subprocess, sys, time, urllib.request
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
for k in ("HF_TOKEN", "EXPLAIN_API_KEY"):
    os.environ[k] = sec.get_secret(k)  # never printed
    print(k, "set")
!pip install -q -U vllm
!rm -rf /kaggle/working/magana && git clone -q https://github.com/jemiridaniel/magana.git /kaggle/working/magana
!git -C /kaggle/working/magana log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Start vLLM (about 5 to 10 minutes: 16 GB download, then load across both T4s)

T4s have no bfloat16, so the model runs in float16 (`--dtype half`), split across the two GPUs (`--tensor-parallel-size 2`). The context is 8,192 tokens, as the Ibà notes use (the model card says "8,092"). LoRA rank 16 matches the Ibà adapter.

In [ ]:
cmd = [
    "vllm", "serve", "NCAIR1/N-ATLaS",
    "--dtype", "half", "--tensor-parallel-size", "2",
    "--max-model-len", "8192", "--gpu-memory-utilization", "0.90",
    "--enable-lora", "--max-lora-rank", "16", "--max-loras", "2",
    "--lora-modules", "iba-atlas=jemiridaniel/Iba-ATLAS-8B-LoRA-Powered-by-Awarri",
    "--api-key", os.environ["EXPLAIN_API_KEY"],
    "--port", "8000",
]
server = subprocess.Popen(cmd, stdout=open("/kaggle/working/vllm.log", "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)
for i in range(90):
    if server.poll() is not None:
        print("vLLM exited, see the log below"); break
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print(f"vLLM ready after {i * 10} s"); break
    except Exception:
        time.sleep(10)
!tail -n 15 /kaggle/working/vllm.log
!nvidia-smi --query-gpu=index,memory.used --format=csv

## 3. Smoke test: base model and Ibà adapter, with Magana's own prompts and number guard

In [ ]:
sys.path.insert(0, "/kaggle/working/magana/backend")
from app import explain, safety
engine_base = explain.OpenAIChatEngine("http://localhost:8000", os.environ["EXPLAIN_API_KEY"], "NCAIR1/N-ATLaS")
page = open("/kaggle/working/magana/benchmark/explain/pages/04_medicine_label.txt").read()
for lang in ("en", "ha", "yo", "ig"):
    t = time.time()
    r = explain.explain(engine_base, page, None, lang, "explain", guided=False, timeout_s=120)
    print(f"--- {lang} {r.code} ({time.time() - t:.1f}s)\n{r.explanation or '(page read instead)'}\n")
# The Ibà adapter is served beside the base model (its own prompts live in iba-natlas).
iba = explain.OpenAIChatEngine("http://localhost:8000", os.environ["EXPLAIN_API_KEY"], "iba-atlas")
print("iba-atlas:", iba.complete("You support health workers.", "Fever 3 days, child 2 years. Danger signs?", 60)[:300])

## 4. Public URL (temporary)

The tunnel forwards to vLLM, which refuses any request without `Authorization: Bearer <EXPLAIN_API_KEY>` (`/health` stays open). Copy the URL it prints, then on your Mac run `scripts/set_explain_url.sh <url>`, which updates the AWS server and restarts the backend.

In [ ]:
!wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /kaggle/working/cloudflared
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)
url = None
for _ in range(30):
    time.sleep(2)
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read())
    if found:
        url = found[0]; break
print("EXPLAIN_URL =", url)
print(f"On your Mac:  scripts/set_explain_url.sh {url}")
# Check it from outside: no key -> 401, /health -> 200.
import requests
print("no key:", requests.post(f"{url}/v1/chat/completions", json={}).status_code,
      "| health:", requests.get(f"{url}/health").status_code)

## 5. Optional: the explanation benchmark against this server

Number-change rate per language, and every output saved for native-speaker rating (`benchmark/explain/README.md`). Download the results folder from the Output panel.

In [ ]:
!cd /kaggle/working/magana && python benchmark/explain/explain_bench.py --engine vllm --url http://localhost:8000 \
    --api-key "$EXPLAIN_API_KEY" --model NCAIR1/N-ATLaS --langs en,ha,yo,ig \
    --out /kaggle/working/bench-base-vllm --note "Kaggle T4 x2, vLLM fp16, base model"

Keep this notebook running while you demo. When you are done, stop the session (top right) so it doesn't use GPU hours.